# NB15 — Stress-test evaluation: hybrid vs neural under polishing and humanizing

Runs both production detectors — **NB10 hybrid** (773) and **NB14 neural** (768) — over the 820
transformed articles from NB13, and over the SAME articles **before** transformation, so every number is
a *degradation* rather than a bare score.

**Label logic (never changes with the transform):**
- *polish*: source is a HUMAN article → still label 0. Predicting AI = **false positive** → **FPR axis**.
- *humanize*: source is an AI article → still label 1. Missing it = **successful evasion** → **TPR axis**.

Statistical features are recomputed with the **exact NB4 + NB6e definitions** and scaled with the
**saved train-fit scalers** (`scaler.pkl`, `scaler11.pkl`) — no refitting. Cell 5 verifies the
implementation by reproducing `vstat16_scaled` on the original articles; if that check fails, every
downstream number is void.

Everything is printed to the log.

## 1 · Config

In [1]:
import os, re, json, gzip, time, pickle, numpy as np, pandas as pd, torch
from collections import Counter

P_DATASET   = "/kaggle/input/datasets/bahaaqassem/aig-and-humang-dataset/dataset.parquet"                 # EDIT
P_GENS      = "/kaggle/input/notebooks/bahaaqassem/nb13-stress-generation-parallel/stress_generations.jsonl"          # EDIT
P_SCALER5   = "/kaggle/input/datasets/bahaaqassem/scalers/scaler.pkl"                       # EDIT (from NB4)
P_SCALER11  = "/kaggle/input/datasets/bahaaqassem/scalers/scaler11.pkl"                     # EDIT (from NB6e)
P_VSTAT16   = "/kaggle/input/datasets/bahaaqassem/aig-16-features/vstat16_scaled.parquet"           # EDIT (verification only)
P_CK_HYBRID = "/kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/ckpt/last.pt"                        # EDIT (NB10)
P_CK_NEURAL = "/kaggle/input/notebooks/bahaaqassem/nb14-neural-only-production/ckpt_neural/last.pt"                 # EDIT (NB14)

MODEL_ID = "CAMeL-Lab/bert-base-arabic-camelbert-msa"
STAT_COLS = ["burstiness","ttr","quote_ratio","function_word_ratio","compressibility"]
K_CHUNKS, MAX_CT, STRIDE = 9, 510, 460
HIDDEN, DROPOUT = 256, 0.0
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[1/9] config loaded | device={DEV} | features={STAT_COLS}", flush=True)

[1/9] config loaded | device=cuda | features=['burstiness', 'ttr', 'quote_ratio', 'function_word_ratio', 'compressibility']


## 2 · Load originals + the 820 transformed articles

In [2]:
df = pd.read_parquet(P_DATASET)
if "article_id" in df.columns: df = df.set_index("article_id")
print(f"[2/9] dataset: {len(df)} articles", flush=True)

rows = [json.loads(l) for l in open(P_GENS, encoding="utf-8")]
G = pd.DataFrame(rows)
G["level"] = G["level"].astype("Float64")
G["gen_id"] = (G.article_id + "__" + G.model + "__" + G.task +
               G.level.map(lambda x: f"__L{int(x)}" if pd.notna(x) else ""))
assert G.gen_id.is_unique, "gen_id not unique"
print(f"[2/9] generations: {len(G)} | tasks {G.task.value_counts().to_dict()}", flush=True)
print(f"[2/9] models: {G.model.value_counts().to_dict()}", flush=True)
print(f"[2/9] polish levels: {G[G.task=='polish'].level.value_counts().sort_index().to_dict()}", flush=True)

# ground-truth label is inherited from the SOURCE article and never changes
G["label"] = df.loc[G.article_id, "label"].to_numpy()
print(f"[2/9] label check — polish sources all human: "
      f"{(G[G.task=='polish'].label==0).all()} | humanize sources all AI: "
      f"{(G[G.task=='humanize'].label==1).all()}", flush=True)

[2/9] dataset: 7101 articles
[2/9] generations: 820 | tasks {'humanize': 410, 'polish': 410}
[2/9] models: {'gemini': 164, 'deepseek': 164, 'qwen': 164, 'claude': 164, 'gpt': 164}
[2/9] polish levels: {np.float64(10.0): 105, np.float64(25.0): 105, np.float64(50.0): 100, np.float64(75.0): 100}
[2/9] label check — polish sources all human: True | humanize sources all AI: True


## 3 · Feature functions — verbatim from NB4 and NB6e

In [3]:
_SENT = re.compile(r'[.!?\u061f\u0964\n]+')
def split_sentences(t): return [s.strip() for s in _SENT.split(t) if s.strip()]
def tokenize_ws(t):     return [x for x in re.split(r'\s+', t.strip()) if x]
AR_DIAC = re.compile(r'[\u064b-\u0652\u0670\u0640]')
def strip_diac(t):      return AR_DIAC.sub('', t)

# --- NB4 ---
def burstiness(text):
    L = np.array([len(tokenize_ws(s)) for s in split_sentences(text)], dtype=float)
    if L.size < 2: return float('nan')
    mu, sigma = L.mean(), L.std()
    return 0.0 if (sigma + mu) == 0 else float((sigma - mu) / (sigma + mu))

def mattr(tokens, window=100):
    if len(tokens) < window:
        return len(set(tokens)) / len(tokens) if tokens else float('nan')
    r = [len(set(tokens[i:i+window]))/window for i in range(len(tokens)-window+1)]
    return float(np.mean(r))
def ttr_feat(text): return mattr(tokenize_ws(text), window=100)

# --- NB6e ---
QUOTE_PAIRS = [('\u00ab','\u00bb'), ('\u201c','\u201d'), ('"','"'), ("'","'")]
def quote_ratio(text):
    toks = tokenize_ws(text)
    if not toks: return float('nan')
    inside = 0
    for op, cl in QUOTE_PAIRS:
        if op == cl:
            parts = text.split(op)
            for k in range(1, len(parts), 2): inside += len(tokenize_ws(parts[k]))
        else:
            for m in re.finditer(re.escape(op)+r'(.*?)'+re.escape(cl), text, flags=re.S):
                inside += len(tokenize_ws(m.group(1)))
    return float(min(inside, len(toks))) / len(toks)

FUNCTION_WORDS = set(['\u0641\u064a','\u0645\u0646','\u0625\u0644\u0649','\u0639\u0644\u0649',
    '\u0639\u0646','\u0645\u0639','\u0628\u064a\u0646','\u0639\u0646\u062f','\u0644\u062f\u0649',
    '\u062e\u0644\u0627\u0644','\u0628\u0639\u062f','\u0642\u0628\u0644','\u0645\u0646\u0630',
    '\u062d\u062a\u0649','\u0625\u0646','\u0623\u0646','\u0623\u0646\u0647','\u0625\u0646\u0647',
    '\u0627\u0644\u0630\u064a','\u0627\u0644\u062a\u064a','\u0647\u0630\u0627','\u0647\u0630\u0647',
    '\u0630\u0644\u0643','\u062a\u0644\u0643','\u0647\u0648','\u0647\u064a','\u0647\u0645',
    '\u0647\u0646','\u0646\u062d\u0646','\u0623\u0646\u0627','\u0643\u0644','\u0628\u0639\u0636',
    '\u063a\u064a\u0631','\u0644\u0627','\u0645\u0627','\u0644\u0645','\u0644\u0646',
    '\u0642\u062f','\u0644\u0642\u062f','\u0623\u064a','\u0623\u064a\u0636\u0627',
    '\u0643\u0645\u0627','\u0644\u0643\u0646','\u0623\u0648','\u062b\u0645','\u0628\u0644',
    '\u0625\u0630','\u0625\u0630\u0627','\u0644\u0623\u0646','\u062d\u064a\u062b'])
def function_word_ratio(text):
    toks = [strip_diac(t) for t in tokenize_ws(text)]
    if not toks: return float('nan')
    return float(sum(1 for t in toks if t in FUNCTION_WORDS)) / len(toks)

def compressibility(text):
    raw = text.encode('utf-8')
    if len(raw) < 200: return float('nan')
    return float(len(gzip.compress(raw, compresslevel=6))) / float(len(raw))

FEAT_FN = {"burstiness":burstiness, "ttr":ttr_feat, "quote_ratio":quote_ratio,
           "function_word_ratio":function_word_ratio, "compressibility":compressibility}
print(f"[3/9] feature functions ready | function-word lexicon size {len(FUNCTION_WORDS)}", flush=True)

[3/9] feature functions ready | function-word lexicon size 50


## 4 · Load the saved train-fit scalers (no refitting)

In [4]:
s5  = pickle.load(open(P_SCALER5, "rb"))
s11 = pickle.load(open(P_SCALER11, "rb"))
print(f"[4/9] scaler.pkl   order={s5['feature_order']}", flush=True)
print(f"[4/9] scaler11.pkl order={s11['feature_order']}", flush=True)

# per-feature (mean, scale, train_median) pulled from whichever pickle owns it
PARAM = {}
for src in (s5, s11):
    order = list(src["feature_order"]); sc = src["scaler"]; med = np.asarray(src["train_median"])
    for f in STAT_COLS:
        if f in order and f not in PARAM:
            i = order.index(f)
            PARAM[f] = (float(sc.mean_[i]), float(sc.scale_[i]), float(med[i]))
missing = [f for f in STAT_COLS if f not in PARAM]
assert not missing, f"no saved scaler params for {missing}"
for f in STAT_COLS:
    m, s, md_ = PARAM[f]
    print(f"[4/9]   {f:22s} mean={m:+.5f} scale={s:.5f} train_median={md_:+.5f}", flush=True)

def compute_scaled(texts, tag=""):
    out = np.empty((len(texts), len(STAT_COLS)), np.float32); t0 = time.time()
    for i, txt in enumerate(texts):
        for j, f in enumerate(STAT_COLS):
            try: v = FEAT_FN[f](str(txt))
            except Exception: v = float('nan')
            m, s, med = PARAM[f]
            if v is None or (isinstance(v, float) and np.isnan(v)): v = med   # train-median imputation
            out[i, j] = (v - m) / s
        if (i+1) % 200 == 0:
            print(f"[4/9]   {tag} featurised {i+1}/{len(texts)} ({time.time()-t0:.0f}s)", flush=True)
    return out
print("[4/9] scaling routine ready", flush=True)

[4/9] scaler.pkl   order=['targeted_ppl', 'burstiness', 'ttr', 'entity_density', 'discourse_coherence']
[4/9] scaler11.pkl order=['quote_ratio', 'quote_variety', 'coord_sub_ratio', 'function_word_ratio', 'pos_entropy', 'passive_ratio', 'clause_depth', 'compressibility', 'char_ngram_repetition', 'zipf_deviation', 'sent_opener_diversity']
[4/9]   burstiness             mean=-0.45286 scale=0.15910 train_median=-0.48272
[4/9]   ttr                    mean=+0.87258 scale=0.02515 train_median=+0.87386
[4/9]   quote_ratio            mean=+0.12442 scale=0.16336 train_median=+0.06370
[4/9]   function_word_ratio    mean=+0.18533 scale=0.02509 train_median=+0.18644
[4/9]   compressibility        mean=+0.33509 scale=0.02357 train_median=+0.33722
[4/9] scaling routine ready


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 5 · VERIFICATION — reproduce vstat16_scaled on original articles

In [5]:
v16 = pd.read_parquet(P_VSTAT16)
if "article_id" in v16.columns: v16 = v16.set_index("article_id")
chk_ids = list(dict.fromkeys(G.article_id.tolist()))[:150]
print(f"[5/9] verifying implementation on {len(chk_ids)} original articles", flush=True)
mine = compute_scaled(df.loc[chk_ids, "text"].tolist(), tag="verify")
theirs = v16.loc[chk_ids, STAT_COLS].to_numpy(np.float32)
ok = True
for j, f in enumerate(STAT_COLS):
    d = np.abs(mine[:, j] - theirs[:, j]); mx = float(d.max()); r = float(np.corrcoef(mine[:,j], theirs[:,j])[0,1])
    flag = "OK " if mx < 1e-3 else ("~  " if r > 0.999 else "BAD")
    if flag == "BAD": ok = False
    print(f"[5/9]   {flag} {f:22s} max|diff|={mx:.6f}  corr={r:.6f}", flush=True)
if ok: print("[5/9] VERIFIED — features reproduce the training pipeline; downstream numbers are valid", flush=True)
else:  print("[5/9] !! MISMATCH — do NOT trust the results below until the definition is reconciled", flush=True)

[5/9] verifying implementation on 150 original articles
[5/9]   OK  burstiness             max|diff|=0.000000  corr=1.000000
[5/9]   OK  ttr                    max|diff|=0.000000  corr=1.000000
[5/9]   OK  quote_ratio            max|diff|=0.000000  corr=1.000000
[5/9]   OK  function_word_ratio    max|diff|=0.000000  corr=1.000000
[5/9]   OK  compressibility        max|diff|=0.000000  corr=1.000000
[5/9] VERIFIED — features reproduce the training pipeline; downstream numbers are valid


## 6 · Features for the transformed articles + their untransformed baselines

In [6]:
base_ids = list(dict.fromkeys(G.article_id.tolist()))
print(f"[6/9] baseline originals: {len(base_ids)} | transformed: {len(G)}", flush=True)
X_base = compute_scaled(df.loc[base_ids, "text"].tolist(), tag="base")
X_gen  = compute_scaled(G.text.tolist(), tag="gen")
base_pos = {a: i for i, a in enumerate(base_ids)}
print(f"[6/9] feature matrices: base {X_base.shape} | gen {X_gen.shape}", flush=True)

drift = X_gen.mean(0) - X_base[[base_pos[a] for a in G.article_id]].mean(0)
print("[6/9] mean feature drift caused by the transforms (scaled units):", flush=True)
for j, f in enumerate(STAT_COLS): print(f"[6/9]   {f:22s} {drift[j]:+.3f}", flush=True)

[6/9] baseline originals: 820 | transformed: 820
[4/9]   base featurised 200/820 (1s)
[4/9]   base featurised 400/820 (2s)
[4/9]   base featurised 600/820 (2s)
[4/9]   base featurised 800/820 (3s)
[4/9]   gen featurised 200/820 (1s)
[4/9]   gen featurised 400/820 (1s)
[4/9]   gen featurised 600/820 (2s)
[4/9]   gen featurised 800/820 (3s)
[6/9] feature matrices: base (820, 5) | gen (820, 5)
[6/9] mean feature drift caused by the transforms (scaled units):
[6/9]   burstiness             +0.127
[6/9]   ttr                    +0.560
[6/9]   quote_ratio            -0.085
[6/9]   function_word_ratio    -0.314
[6/9]   compressibility        +0.573


## 7 · Chunk both sets (K=9, 510/460 — the training contract)

In [7]:
!pip install -q transformers
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
CLS, SEP, PAD = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id
CHUNK_LEN = MAX_CT + 2

def chunk_all(texts, tag):
    CH = np.full((len(texts), K_CHUNKS, CHUNK_LEN), PAD, np.int32)
    NC = np.zeros(len(texts), np.int8); t0 = time.time()
    for i, t in enumerate(texts):
        ids = tok(str(t), add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
        wins = [ids[k:k+MAX_CT] for k in range(0, len(ids), STRIDE)][:K_CHUNKS] or [ids[:MAX_CT]]
        NC[i] = len(wins)
        for j, w in enumerate(wins): CH[i, j] = [CLS]+w+[SEP]+[PAD]*(CHUNK_LEN-2-len(w))
        if (i+1) % 300 == 0: print(f"[7/9]   {tag} chunked {i+1}/{len(texts)} ({time.time()-t0:.0f}s)", flush=True)
    return CH, NC

CH_base, NC_base = chunk_all(df.loc[base_ids, "text"].tolist(), "base")
CH_gen,  NC_gen  = chunk_all(G.text.tolist(), "gen")
print(f"[7/9] chunks ready | base {CH_base.shape} | gen {CH_gen.shape}", flush=True)

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[7/9]   base chunked 300/820 (1s)
[7/9]   base chunked 600/820 (1s)
[7/9]   gen chunked 300/820 (1s)
[7/9]   gen chunked 600/820 (1s)
[7/9] chunks ready | base (820, 9, 512) | gen (820, 9, 512)


## 8 · Load both detectors and predict

In [8]:
import torch.nn as nn
from transformers import AutoModel
from torch.utils.data import DataLoader, Dataset

class Net(nn.Module):
    def __init__(self, use_stat, stat_dim=5):
        super().__init__()
        self.use_stat = use_stat
        self.enc = AutoModel.from_pretrained(MODEL_ID)
        h = self.enc.config.hidden_size
        fused = h + (stat_dim if use_stat else 0)
        self.register_buffer("mu", torch.zeros(fused)); self.register_buffer("sd", torch.ones(fused))
        self.head = nn.Sequential(nn.Linear(fused, HIDDEN), nn.LayerNorm(HIDDEN), nn.ReLU(),
                                  nn.Dropout(DROPOUT), nn.Linear(HIDDEN, 2))
    def encode(self, ids, nch):
        B,K,L = ids.shape; flat = ids.view(B*K, L); att = (flat != PAD).long()
        cls = self.enc(input_ids=flat, attention_mask=att).last_hidden_state[:,0,:].view(B,K,-1)
        m = (torch.arange(K, device=ids.device)[None,:] < nch[:,None]).float().unsqueeze(-1)
        return (cls*m).sum(1)/m.sum(1).clamp(min=1)
    def forward(self, ids, nch, stat=None):
        v = self.encode(ids, nch)
        if self.use_stat: v = torch.cat([v, stat], 1)
        v = (v - self.mu)/self.sd
        return self.head(v)

class DS(Dataset):
    def __init__(self, CH, NC, X): self.CH, self.NC, self.X = CH, NC, X
    def __len__(self): return len(self.CH)
    def __getitem__(self, i):
        return (torch.from_numpy(self.CH[i].astype(np.int64)), int(self.NC[i]),
                torch.from_numpy(self.X[i]))
def collate(b):
    return (torch.stack([x[0] for x in b]), torch.tensor([x[1] for x in b]),
            torch.stack([x[2] for x in b]))

@torch.no_grad()
def predict(model, CH, NC, X, tag):
    model.eval(); P=[]; t0=time.time()
    dl = DataLoader(DS(CH,NC,X), batch_size=8, collate_fn=collate)
    for bi,(ids,nch,st) in enumerate(dl,1):
        ids,nch,st = ids.to(DEV), nch.to(DEV), st.to(DEV)
        with torch.amp.autocast('cuda'):
            P.append(torch.softmax(model(ids,nch,st),1)[:,1].float().cpu().numpy())
        if bi % 25 == 0: print(f"[8/9]   {tag} {bi*8}/{len(CH)} ({time.time()-t0:.0f}s)", flush=True)
    return np.concatenate(P)

PRED = {}
for name, ck, use_stat in [("hybrid", P_CK_HYBRID, True), ("neural", P_CK_NEURAL, False)]:
    print(f"[8/9] loading {name} from {ck}", flush=True)
    m = Net(use_stat).to(DEV)
    sd = torch.load(ck, map_location=DEV, weights_only=False)["model"]
    m.load_state_dict(sd)
    print(f"[8/9]   {name}: fused dim {m.mu.numel()} | standardizer |mu|={float(m.mu.abs().mean()):.4f}", flush=True)
    PRED[(name,"base")] = predict(m, CH_base, NC_base, X_base, f"{name}/base")
    PRED[(name,"gen")]  = predict(m, CH_gen,  NC_gen,  X_gen,  f"{name}/gen")
    del m; torch.cuda.empty_cache()
    print(f"[8/9] {name} done", flush=True)

[8/9] loading hybrid from /kaggle/input/notebooks/bahaaqassem/nb10-trackb-joint-finetune/ckpt/last.pt


pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[8/9]   hybrid: fused dim 773 | standardizer |mu|=0.4729
[8/9]   hybrid/base 200/820 (12s)
[8/9]   hybrid/base 400/820 (24s)
[8/9]   hybrid/base 600/820 (36s)
[8/9]   hybrid/base 800/820 (49s)
[8/9]   hybrid/gen 200/820 (13s)
[8/9]   hybrid/gen 400/820 (28s)
[8/9]   hybrid/gen 600/820 (43s)
[8/9]   hybrid/gen 800/820 (58s)
[8/9] hybrid done
[8/9] loading neural from /kaggle/input/notebooks/bahaaqassem/nb14-neural-only-production/ckpt_neural/last.pt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-msa
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[8/9]   neural: fused dim 768 | standardizer |mu|=0.4760
[8/9]   neural/base 200/820 (14s)
[8/9]   neural/base 400/820 (29s)
[8/9]   neural/base 600/820 (45s)
[8/9]   neural/base 800/820 (59s)
[8/9]   neural/gen 200/820 (14s)
[8/9]   neural/gen 400/820 (28s)
[8/9]   neural/gen 600/820 (42s)
[8/9]   neural/gen 800/820 (57s)
[8/9] neural done


## 9 · Results — FPR under polishing, TPR under humanizing, vs the untransformed baseline

In [9]:
G = G.reset_index(drop=True)
bidx = np.array([base_pos[a] for a in G.article_id])
for name in ("hybrid","neural"):
    G[f"p_{name}_base"] = PRED[(name,"base")][bidx]
    G[f"p_{name}_gen"]  = PRED[(name,"gen")]

def rate(p, label):           # fraction predicted AI
    return 100.0*float((p >= 0.5).mean()) if len(p) else float('nan')

print("="*100, flush=True)
print("[9/9] A · POLISHING — human articles, so 'predicted AI' = FALSE POSITIVE (lower is better)", flush=True)
pol = G[G.task=="polish"]
rowsA=[]
for lvl in sorted(pol.level.dropna().unique()):
    s = pol[pol.level==lvl]
    rowsA.append({"level":int(lvl), "n":len(s),
                  "FPR_base_hyb":round(rate(s.p_hybrid_base,0),2), "FPR_pol_hyb":round(rate(s.p_hybrid_gen,0),2),
                  "FPR_base_neu":round(rate(s.p_neural_base,0),2), "FPR_pol_neu":round(rate(s.p_neural_gen,0),2)})
A = pd.DataFrame(rowsA)
A["dFPR_hyb"] = (A.FPR_pol_hyb - A.FPR_base_hyb).round(2)
A["dFPR_neu"] = (A.FPR_pol_neu - A.FPR_base_neu).round(2)
print(A.to_string(index=False), flush=True)

print("\n[9/9] A · by polishing model:", flush=True)
print(pol.groupby("model").apply(lambda s: pd.Series({
    "n":len(s), "FPR_hyb":round(rate(s.p_hybrid_gen,0),2), "FPR_neu":round(rate(s.p_neural_gen,0),2)}),
    include_groups=False).to_string(), flush=True)

print("\n"+"="*100, flush=True)
print("[9/9] B · HUMANIZING — AI articles, so 'predicted AI' = TPR retained (higher is better)", flush=True)
hum = G[G.task=="humanize"]
rowsB=[{"scope":"ALL","n":len(hum),
        "TPR_base_hyb":round(rate(hum.p_hybrid_base,1),2), "TPR_hum_hyb":round(rate(hum.p_hybrid_gen,1),2),
        "TPR_base_neu":round(rate(hum.p_neural_base,1),2), "TPR_hum_neu":round(rate(hum.p_neural_gen,1),2)}]
for mdl in sorted(hum.model.unique()):
    s = hum[hum.model==mdl]
    rowsB.append({"scope":mdl,"n":len(s),
        "TPR_base_hyb":round(rate(s.p_hybrid_base,1),2), "TPR_hum_hyb":round(rate(s.p_hybrid_gen,1),2),
        "TPR_base_neu":round(rate(s.p_neural_base,1),2), "TPR_hum_neu":round(rate(s.p_neural_gen,1),2)})
B = pd.DataFrame(rowsB)
B["dTPR_hyb"] = (B.TPR_hum_hyb - B.TPR_base_hyb).round(2)
B["dTPR_neu"] = (B.TPR_hum_neu - B.TPR_base_neu).round(2)
print(B.to_string(index=False), flush=True)

print("\n"+"="*100, flush=True)
print("[9/9] C · TPR at a low-FPR operating point (action #7)", flush=True)
for name in ("hybrid","neural"):
    clean_h = G[G.task=="polish"][f"p_{name}_base"].to_numpy()      # untransformed human scores
    for target in (0.01, 0.05):
        thr = float(np.quantile(clean_h, 1-target))
        hb = float((hum[f"p_{name}_base"].to_numpy() >= thr).mean())*100
        hg = float((hum[f"p_{name}_gen"].to_numpy()  >= thr).mean())*100
        print(f"[9/9]   {name:6s} @FPR={target*100:.0f}% (thr={thr:.4f}): "
              f"TPR base {hb:.2f}% -> humanized {hg:.2f}%  (drop {hb-hg:+.2f})", flush=True)

G.drop(columns=["text"]).to_parquet("/kaggle/working/nb15_stress_results.parquet", index=False)
print("\n[9/9] saved -> /kaggle/working/nb15_stress_results.parquet", flush=True)
print("[9/9] READ: compare dFPR_hyb vs dFPR_neu and dTPR_hyb vs dTPR_neu — a POSITIVE gap for the", flush=True)
print("[9/9]       hybrid is the first evidence that statistical fusion survives transformation.", flush=True)

[9/9] A · POLISHING — human articles, so 'predicted AI' = FALSE POSITIVE (lower is better)
 level   n  FPR_base_hyb  FPR_pol_hyb  FPR_base_neu  FPR_pol_neu  dFPR_hyb  dFPR_neu
    10 105          0.95         2.86          0.95         3.81      1.91      2.86
    25 105          0.00         1.90          0.00         6.67      1.90      6.67
    50 100          0.00        10.00          0.00        12.00     10.00     12.00
    75 100          1.00        21.00          0.00        25.00     20.00     25.00

[9/9] A · by polishing model:
             n  FPR_hyb  FPR_neu
model                           
claude    82.0    10.98    14.63
deepseek  82.0     7.32    13.41
gemini    82.0    21.95    26.83
gpt       82.0     1.22     0.00
qwen      82.0     2.44     3.66

[9/9] B · HUMANIZING — AI articles, so 'predicted AI' = TPR retained (higher is better)
   scope   n  TPR_base_hyb  TPR_hum_hyb  TPR_base_neu  TPR_hum_neu  dTPR_hyb  dTPR_neu
     ALL 410         100.0        94.15       